# 04. Explainability Analysis & Feature Attributions

**Course:** URP4201 Research Methodology

Global TreeSHAP feature importance beeswarm analysis and local counterfactual occlusion attributions.

In [1]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "AquaSense AI" else Path(".").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import joblib
import numpy as np
import pandas as pd
from src.aquasense.explain import explain_sample_occlusion, FEATURE_NAMES

bundle = joblib.load("models/model_bundle.joblib")
clf = bundle["model"]
medians = np.array(bundle["training_medians"])

print("Loaded Model Pipeline:", type(clf).__name__)
print("Baseline Training Medians for Occlusion:")
for feat, val in zip(FEATURE_NAMES, medians):
    print(f"  {feat:30}: {val:.4f}")

Loaded Model Pipeline: CalibratedClassifierCV
Baseline Training Medians for Occlusion:
  pH                            : 7.1520
  Turbidity                     : 2.8950
  Temperature                   : 22.1000
  Dissolved Oxygen              : 5.5800
  Biochemical Oxygen Demand     : 2.4100
  Lead                          : 0.0061
  Mercury                       : 0.0004
  Arsenic                       : 0.0057


In [2]:
# Test Case 1: Nominal Potable Water Sample
sample_good = np.array([7.2, 2.5, 22.0, 7.0, 1.5, 0.003, 0.0002, 0.003])
exp_good = explain_sample_occlusion(clf, sample_good, medians)
print(f"Predicted Class Index: {exp_good['predicted_class']} (Base Probability: {exp_good['base_probability']:.4f})")
pd.DataFrame(list(exp_good["attributions"].items()), columns=["Water Parameter", "Probability Delta (Delta P)"])

Predicted Class Index: 0 (Base Probability: 0.9846)


,Water Parameter,Probability Delta (Delta P)
0,pH,0.0
1,Turbidity,0.0
2,Temperature,0.0
3,Dissolved Oxygen,0.0
4,Biochemical Oxygen Demand,0.0
5,Lead,0.0
6,Mercury,0.0
7,Arsenic,0.0


In [3]:
# Test Case 2: Heavily Polluted Industrial Effluent Sample
sample_polluted = np.array([4.8, 38.0, 28.0, 2.2, 18.0, 0.05, 0.004, 0.03])
exp_polluted = explain_sample_occlusion(clf, sample_polluted, medians)
print(f"Predicted Class Index: {exp_polluted['predicted_class']} (Base Probability: {exp_polluted['base_probability']:.4f})")
pd.DataFrame(list(exp_polluted["attributions"].items()), columns=["Water Parameter", "Probability Delta (Delta P)"])

Predicted Class Index: 2 (Base Probability: 0.9794)


,Water Parameter,Probability Delta (Delta P)
0,Mercury,0.0270
1,Arsenic,0.0066
2,Biochemical Oxygen Demand,0.0057
3,Turbidity,0.0056
4,Lead,0.0056
5,pH,0.0019
6,Dissolved Oxygen,0.0010
7,Temperature,0.0000


In [4]:
# Cross-Fold Feature Ranking Stability Audit
with open("results/metrics.json", "r", encoding="utf-8") as f:
    stability = json.load(f)["feature_rank_stability"]

print("Mean Spearman Rank Correlation across 5 Folds:", stability["mean_spearman_rank_correlation"])
print("Ranking Stability Assessment:", stability["stability_level"])

Mean Spearman Rank Correlation across 5 Folds: 0.9505
Ranking Stability Assessment: High
